# CPEN355 Cat and Dog Breed Classification - Google Colab

Complete end-to-end pipeline from setup to training and inference.

Run cells in order to:
1. Setup environment
2. Install dependencies
3. Clone/setup repository
4. Train model
5. Evaluate model
6. Make predictions

In [ ]:
import os
import subprocess

# Check if running on Colab
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")


def run_cmd(cmd):
    """Run a command and stream stdout/stderr as it executes."""
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    return process.returncode

In [ ]:
if IN_COLAB:
    run_cmd(['git', 'clone', 'https://github.com/erictannant/cpen355-project.git'])
    os.chdir('cpen355-project')
    print("Repository cloned")

    # cd into the cloned github repo
    os.chdir('/content/cpen355-project')
    print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Data Preparation

Change config file for different breeds

In [ ]:
run_cmd(['python', 'scripts/download_data.py', '--config', 'configs/resnet50.yaml'])
run_cmd(['python', 'scripts/prepare_data.py', '--config', 'configs/resnet50.yaml'])

In [ ]:
run_cmd(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print("Dependencies installed")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import json
import yaml
from sklearn.metrics import confusion_matrix, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Train - CNN Model

In [ ]:
run_cmd(['python', '-m', 'src.train', '--config', 'configs/cnn.yaml'])

## Train - ResNet18

In [ ]:
run_cmd(['python', '-m', 'src.train', '--config', 'configs/resnet18.yaml'])

## Train - ResNet50

In [ ]:
run_cmd(['python', '-m', 'src.train', '--config', 'configs/resnet50.yaml'])

## Fine Tune CNN

In [ ]:
run_cmd(['python', '-m', 'src.fine_tune', '--config', 'configs/cnn.yaml', '--objective', 'val_acc', '--max-trials', '30'])

## Evaluate Model

This allows measuring the performance of the model.

In [ ]:
run_cmd(['python', '-m', 'src.evaluate', '--config', 'configs/cnn.yaml'])

In [ ]:
run_cmd(['python', '-m', 'src.evaluate', '--config', 'configs/resnet18.yaml'])

In [ ]:
run_cmd(['python', '-m', 'src.evaluate', '--config', 'configs/resnet50.yaml'])

## Inference

Set an image path and run the model-specific command cells below.

Replace the example image path with a different path in your runtime if you want to test other images. Run the evaluate function above if you want large scale testing.

In [ ]:
run_cmd([
    'python', '-m', 'src.infer',
    '--config', 'configs/cnn.yaml',
    '--checkpoint', 'outputs/checkpoints/cnn/best.pt',
    '--image', 'data/raw/images/pug_82.jpg',
    '--top-k', '3'
])

## Inference (ResNet18)

In [ ]:
run_cmd([
    'python', '-m', 'src.infer',
    '--config', 'configs/resnet18.yaml',
    '--checkpoint', 'outputs/checkpoints/resnet18/best.pt',
    '--image', 'data/raw/images/pug_82.jpg',
    '--top-k', '3'
])

## Inference (ResNet50)

In [ ]:
run_cmd([
    'python', '-m', 'src.infer',
    '--config', 'configs/resnet50.yaml',
    '--checkpoint', 'outputs/checkpoints/resnet50/best.pt',
    '--image', 'data/raw/images/pug_82.jpg',
    '--top-k', '3'
])